# 01 — Baseline Model

**Amaç:** MNIST üzerinde baseline MLP modelini kurmak, epoch bazında training/test loss ve accuracy değerlerini kaydetmek ve ödevde istenen baseline grafiklerini üretmek.

Architecture: `784 → 128 → ReLU → 64 → ReLU → 10`. Bu notebook baseline deneyinin ölçülebilir kayıtlarını üretir.

In [ ]:
# Temel kütüphaneleri içe aktarırız; veri, model, süre ve grafik ölçümlerinde kullanılır.
import os, sys, time, json, random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# Seed'i sabitleriz; aynı başlangıç koşullarını mümkün olduğunca yeniden üretmek için kullanılır.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 5
transform = transforms.ToTensor()
train = datasets.MNIST('data', train=True, download=True, transform=transform)
test = datasets.MNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)
print('train:', len(train), 'test:', len(test))

In [ ]:
# Loss ve accuracy hesaplar; modelin her epoch sonunda hem training hem test davranışını ölçer.
def evaluate_metrics(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

In [ ]:
# Modeli eğitir ve epoch bazında train/test loss ve accuracy değerlerini kaydeder.
model = build_model(SEED).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()
history = {'train_loss': [], 'train_accuracy': [], 'test_loss': [], 'test_accuracy': []}
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
    tr_loss, tr_acc = evaluate_metrics(model, train_loader)
    te_loss, te_acc = evaluate_metrics(model, test_loader)
    history['train_loss'].append(tr_loss)
    history['train_accuracy'].append(tr_acc)
    history['test_loss'].append(te_loss)
    history['test_accuracy'].append(te_acc)
    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | test_loss={te_loss:.4f} | test_acc={te_acc:.4f}')
training_seconds = time.time() - start
final_accuracy = history['test_accuracy'][-1]
os.makedirs('../results', exist_ok=True)
torch.save(model.state_dict(), '../results/baseline_model.pt')
with open('../results/baseline_history.json', 'w', encoding='utf-8') as f: json.dump(history, f, indent=2)
config = {'seed': SEED, 'architecture': '784-128-ReLU-64-ReLU-10', 'optimizer': 'Adam', 'learning_rate': LR, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'accuracy': final_accuracy, 'training_seconds': training_seconds}
with open('../results/baseline_config.json', 'w', encoding='utf-8') as f: json.dump(config, f, indent=2)
print('final_accuracy:', final_accuracy)
print('training_seconds:', round(training_seconds, 2))

In [ ]:
# Loss grafiği; training ve test/validation davranışını epoch boyunca karşılaştırır.
epochs = range(1, EPOCHS + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs, history['train_loss'], marker='o', label='Training Loss')
plt.plot(epochs, history['test_loss'], marker='o', label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Baseline Training vs Test Loss')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
# Accuracy grafiği; training ve test accuracy'nin epoch boyunca değişimini gösterir.
plt.figure(figsize=(7, 4))
plt.plot(epochs, history['train_accuracy'], marker='o', label='Training Accuracy')
plt.plot(epochs, history['test_accuracy'], marker='o', label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Baseline Training vs Test Accuracy')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

## Baseline kayıt
Epoch bazındaki gerçek loss ve accuracy değerleri `results/baseline_history.json` dosyasına kaydedilir. Final accuracy, süre ve deney parametreleri `notes/experiment_log.md` dosyasına aktarılabilir.